# Emotion Detection & Topic Modelling

## Functions

In [ ]:
import json
def split_list_into_chunks(data, chunk_size=50000):
    """
    Splits a list into chunks of specified size.
    
    Args:
        data (list): The list to split.
        chunk_size (int): Number of elements per chunk.
        
    Returns:
        List of chunks (lists).
    """
    return [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)]

def find_duplicates_set(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
        else:
            seen.add(item)
    return list(duplicates)

def remove_duplicates_dicts(data):
    seen = set()
    unique = []
    for item in data:
        item_str = json.dumps(item, sort_keys=True)  # Convert dict to string
        if item_str not in seen:
            seen.add(item_str)
            unique.append(item)
    return unique


# Convert the response content (JSONL) to a list of dictionaries
def jsonl_to_list(response):
    # Decode the response content if it's in bytes
    content = response.text if hasattr(response, 'text') else response.decode('utf-8')
    
    # Split by newlines and load each line as a dictionary
    return [json.loads(line) for line in content.strip().split('\n') if line]

In [ ]:
from openai.lib._pydantic import to_strict_json_schema
from pydantic import BaseModel, Field, create_model
from typing import List, Literal

def create_emotion_topic_model():
    """
    EmotionTopicModel represents a model containing a list of dictionaries encapsulating topics, emotions, and explanations related to a given text.
    """
    class EmotionTopicItem(BaseModel):
        """
        EmotionTopicItem represents a single topic with its name and an optional explanation.
        """
        entity: str = Field(default="", description="The entity of the text in English if applicable.")
        topic: str = Field(..., description="The topic of the text in English.")
        emotion: Literal[
            "anger", "anticipation", "disgust", "fear", "joy", "sadness", "surprise", "trust"
        ] = Field(..., description="The Plutchik emotion associated with the topic in English")
        topic_explanation: str = Field(..., description="Short explanation of why the topic was chosen in English.")
        emotion_explanation: str = Field(..., description="Short explanation of why the emotion was chosen in English.")
        
        class Config:
            extra = "forbid"  # Disallow additional properties

    fields = {
        "emotopic": (List[EmotionTopicItem], Field(..., description="List of topics related to the text in English", max_items=5)),
        "bot": (bool, Field(default=False, description="Indicates if the response was generated by a bot. Default is False.")),
        "genai": (bool, Field(default=False, description="Indicates if the response was generated by a generative AI model. Default is False.")),
    }

    EmoTopicGroup = create_model("TopicGroup", **fields)
    EmoTopicGroup.__doc__ = """
        EmotionTopicModel class representing a model containing a list of dictionaries encapsulating topics, emotions, and explanations related to a given text.
        The model is designed to handle a list of topics, each associated with a specific emotion from Plutchik's wheel of emotions.
        The model also includes flags to indicate if the response was generated by a bot or a generative AI model.
    """

    # Convert to strict schema
    schema = to_strict_json_schema(EmoTopicGroup)
    schema["additionalProperties"] = False
    if "$defs" in schema and "EmotionTopicItem" in schema["$defs"]:
        schema["$defs"]["EmotionTopicItem"]["additionalProperties"] = False

    return {
        key: schema[key] for key in ["additionalProperties", "$defs", "description", "properties", "title", "type", "required"]
    }

In [ ]:
def create_system_prompt(search_term: str) -> str:
    return f"""You are an advanced language model designed to analyze texts and extract structured information.

Context:
The text you will review are YouTube transcripts or comments on videos relating to the US tariff policy of 2025.
The search term used was {search_term}. This may not be directly related to the entity discussed in the text and should be used only as a reference for context.

Task:
Given a text, identify and extract the following details:

Entity – The entity of the text in English if applicable. If not applicable, it should be an empty string.
Topics – List the topics discussed in the text. Maximum number of topics is 5.
Emotions – For each topic, identify the emotions expressed (anger, anticipation, disgust, fear, joy, sadness, surprise, trust).
Topic Explanation – Concisely explain the reason behind the topic relating to the text. The explanations should be based on the text.
Emotion Explanation – Concisely explain the reason behind the emotion relating to the topic. The reasons should be based on the text, not inferred from the topic.

If topic has a neutral emotion, do not include it in the response.
"""

In [ ]:
def create_batch_payload(
    data: list,
    filename: str,
    search_term: str ,
    model: str = 'gpt-4.1-mini-2025-04-14',
    schema: str = create_emotion_topic_model()
    ):
    """
    Create a batch payload for the OpenAI API.

    Args:
        data (list): The list of texts to be analysed.
        filename (str): The name of the file to be processed.
        system_prompt (str): The system prompt to be used for the API.
        model (str): The model to be used for the API.
        response_format (str): The response format for the API.

    Returns:
        list: The batch payload.
    """

    # Create the batch payload
    payload = []
    for text in data:
        if isinstance(text['text'], str):
            payload.append(
                {
                    "custom_id": text['id'], 
                    "method": "POST", 
                    "url": "/v1/chat/completions", 
                    "body": {
                        "model": model, 
                        "messages": [
                            {
                                "role": "system", 
                                "content": create_system_prompt(search_term)
                            }, 
                            {
                                "role": "user", 
                                "content": text['text']
                            }
                        ],
                        "response_format": {
                            "type": "json_schema",
                            "json_schema": {
                                "name": "topic_emotion_analysis",
                                "schema": schema,
                                "strict": True
                            }
                        }
                    }
                }
            )

    with open(filename, 'w') as file:
        for item in payload:
            json_line = json.dumps(item)
            file.write(json_line + '\n')

In [ ]:
import os
import json
from google.cloud import storage
import pandas as pd
from io import StringIO

def load_from_gcs(bucket_name, blob_name):
    """
    Downloads a CSV file from GCS and loads it into a Python variable.

    Args:
        bucket_name (str): GCS bucket name.
        blob_name (str): Path to the CSV file in GCS.

    Returns:
        dict or list: Parsed JSON content.
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    json_content = blob.download_as_text()
    df = pd.read_csv(StringIO(json_content))
    return df.to_dict(orient='records')

def upload_to_gcs(bucket_name, data, destination_blob_name):
    """
    Uploads a file to Google Cloud Storage and deletes the local file afterward.
    
    Args:
        bucket_name (str): GCS bucket name.
        data (dict): Data to be uploaded.
        destination_blob_name (str): Path to the file in GCS.
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)

    # Convert data to JSON string
    json_data = json.dumps(data)

    # Upload JSON string directly
    blob.upload_from_string(json_data, content_type='application/json')

def upload_and_delete_local_file(bucket_name, source_file_name, destination_blob_name):
    """Uploads a file to Google Cloud Storage and deletes the local file afterward."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)

    # Upload the file to GCS
    blob.upload_from_filename(source_file_name)

    # Delete the local file
    if os.path.exists(source_file_name):
        os.remove(source_file_name)
    else:
        pass

def list_from_gcs(bucket_name, folder_prefix, filename=None):
    storage_client = storage.Client()
    blobs = storage_client.list_blobs(bucket_name, prefix=folder_prefix)
    
    if filename:
        return [blob.name for blob in blobs if filename in blob.name]
    else:
        return [blob.name for blob in blobs]

In [ ]:
import os
import json
from openai import OpenAI
from tqdm.notebook import tqdm

class openai_batch_process:
    def __init__(
        self,
        key: str,
        query: str,
        filename: str,
        bucket_name: str = 'youtube-us-tariffs2'
        ):
        self.client = OpenAI(api_key=key)
        self.query = query
        self.filename = filename.replace('csv', 'json')
        self.bucket_name = bucket_name
        
        self.data = None
        self.paths = None
        self.file_ids = None
        self.batch_ids = None
        self.output_file_ids = None
        self.output = None

    def download_from_gcs(
        self,
        id_field: str,
        text_field: str,
        filter_ids: list = None,
        filter_col: str = None,
        ):
        # Get list of files
        blobnames = list_from_gcs(self.bucket_name, '-'.join(self.query.split()).lower(), self.filename.replace('json', 'csv'))

        assert len(blobnames) > 0, f"No files found!"
        
        # Download from GCS
        all_data = []

        for blob in tqdm(blobnames, desc="Downloading blobs", unit="blob", total=len(blobnames)):
            data = load_from_gcs(self.bucket_name, blob_name=blob)
            if data:
                # Filter if filter_ids
                if filter_ids:
                    filter_data = [{'id': x[id_field], 'text': x[text_field]} for x in data if x[filter_col] in filter_ids]
                else:
                    filter_data = [{'id': x[id_field], 'text': x[text_field]} for x in data]
                all_data += filter_data

        # Remove Duplicates
        self.data = remove_duplicates_dicts(all_data)

    def create_payload(
        self,
        chunk_size: int = 50_000
        ):  
        # Split data
        data_splits = split_list_into_chunks(self.data, chunk_size=chunk_size)
        paths = []

        if len(data_splits) == 1:
            path = f"data/{'-'.join(self.query.split()).lower()}_{self.filename.replace('.json', '')}.jsonl"
            create_batch_payload(data=data_splits[0], filename=path, search_term=f"'{self.query}'")

            size_bytes = os.path.getsize(path)
            size_kb = size_bytes / 1024
            size_mb = size_kb / 1024

            for i in tqdm([1], desc="Creating payloads", unit="payload", total=len(data_splits)):
                assert size_mb < 200, f"File size of {size_mb:.2f} MB exceeds maximum of 200 MB. Reduce the chunk size."

            paths.append(path)

        else:
            for i, lst in enumerate(tqdm(data_splits, desc="Creating payloads", unit="payload", total=len(data_splits))):
                path = f"data/{'-'.join(self.query.split()).lower()}_{self.filename.replace('.json', '')}_{i}.jsonl"
                create_batch_payload(data=lst, filename=path, search_term=f"'{self.query}'")

                size_bytes = os.path.getsize(path)
                size_kb = size_bytes / 1024
                size_mb = size_kb / 1024

                assert size_mb < 200, f"File size of {size_mb:.2f} MB exceeds maximum of 200 MB. Reduce the chunk size."

                paths.append(path)

        self.paths = paths

    def create_batch(
        self,
        create_file: bool = True,
        create_batch: bool = True
        ):
        
        if create_file:
            file_ids = []
            
            for path in tqdm(self.paths, desc="Creating files", unit="file", total=len(self.paths)):
                batch_input_file = self.client.files.create(
                    file=open(path, "rb"),
                    purpose="batch"
                )
                file_ids.append(batch_input_file.id)
                
            self.file_ids = file_ids
        
        if create_batch and len(self.file_ids) > 0:
            batch_ids = []

            for path, file_id in tqdm(zip(self.paths, self.file_ids), desc="Creating batches", unit="batch", total=len(self.file_ids)):
                batch_details = self.client.batches.create(
                    input_file_id=file_id,
                    endpoint="/v1/chat/completions",
                    completion_window="24h",
                    metadata={
                        "description": path
                    }
                )
                batch_ids.append(batch_details.id)
            
            self.batch_ids = batch_ids

        if create_file and not create_batch:
            metadafile = f"data/files_{'-'.join(self.query.split()).lower()}_{self.filename}"

            with open(metadafile, 'w') as json_file:
                json.dump(
                    {
                        "paths": self.paths,
                        "file_ids": self.file_ids
                    },
                    json_file
                )

            print(f"IDs saved as {metadafile}")
        
        else:
            metadafile = f"data/metadata_{'-'.join(self.query.split()).lower()}_{self.filename}"

            with open(metadafile, 'w') as json_file:
                json.dump(
                    {
                        "paths": self.paths,
                        "file_ids": self.file_ids,
                        "batch_ids": self.batch_ids
                    },
                    json_file
                )

            print(f"IDs saved as {metadafile}")
        
    def retrieve_batch(
        self,
        batch_ids: list = None,
        print_status: bool = True
        ):
        
        if batch_ids:
            id_list = batch_ids
        else:
            try:
                id_list = self.batch_ids
            except:
                print("Provide list of Batch IDs")
        
        output_file_ids = []
        
        for batch_id in id_list:
            batch = self.client.batches.retrieve(batch_id)
            
            if print_status:
                print(f"{batch}\n")
            
            output_file_ids.append(batch.output_file_id)
            
        if output_file_ids and not all(item is None for item in output_file_ids):
            self.output_file_ids = output_file_ids
    
    def cancel_batch(
        self,
        batch_ids: list = None
        ):
        
        if batch_ids:
            id_list = batch_ids
        else:
            id_list = self.batch_ids
        
        for batch_id in tqdm(id_list, desc="Cancelling Batches", unit="batch", total=len(id_list)):
            self.client.batches.cancel(batch_id)
    
    def download_output(
        self,
        metadata_path: str = None
        ):
        
        if metadata_path is None:
            metadata_path = f"data/metadata_{'-'.join(self.query.split()).lower()}_{self.filename}"
    
        # Check batch ids exist
        if not self.output_file_ids:
            # Load existing file ids
            with open(metadata_path, "r") as file:
                data = json.load(file)
                
                self.file_ids = data['file_ids']
                self.batch_ids = data['batch_ids']
                
            # Get output file IDs
            self.retrieve_batch(print_status=False)
        
        for i, output_file_id in enumerate(tqdm(self.output_file_ids, desc="Downloading Outputs", unit="file", total=len(self.batch_ids))):
            # Download processed data
            file_response = self.client.files.content(output_file_id)
            
            output_file_name = f"data/{'-'.join(self.query.split()).lower()}_{self.filename.replace('.json', '')}_{i}_processed.jsonl"
            with open(output_file_name, "w", encoding="utf-8") as file:
                file.write(file_response.text)
            
            print(f"Output saved as {output_file_name}")
        
    def process_batch(
        self,
        id_field: str,
        text_field: str,
        filter_ids: list = None,
        filter_col: str = None,
        chunk_size: int = 50_000,
        download: bool = True,
        payload: bool = True,
        batch: bool = True
        ):
        # Download files
        if download:
            self.download_from_gcs(
                id_field=id_field,
                text_field=text_field,
                filter_ids=filter_ids,
                filter_col=filter_col,
            )
        # Create payloads
        if payload:
            self.create_payload(
                chunk_size=chunk_size,
            )
        # Upload files and create batches
        if batch:
            self.create_batch()
    
    def cleanup_openai(
        self,
        input_files: bool = False,
        batches: bool = False,
        output_files: bool = False
        ):
        
        assert all(isinstance(x, bool) for x in [input_files, batches, output_files]), "Boolean inputs only (True or False)"
        assert any([input_files, batches, output_files]), "No item selected for cleanup"
        
        if input_files:
            for file_id in tqdm(self.file_ids, desc="Deleting Input Files", unit="file", total=len(self.file_ids)):
                self.client.files.delete(file_id)
        if batches:
            for batch_id in tqdm(self.batch_ids, desc="Deleting Batches", unit="batch", total=len(self.batch_ids)):
                self.client.batches.cancel(batch_id)
        if output_files:
            for file_id in tqdm(self.output_file_ids, desc="Deleting Output Files", unit="file", total=len(self.output_file_ids)):
                self.client.files.delete(file_id)

In [ ]:
from keys import OPENAI_KEY
client = OpenAI(api_key=OPENAI_KEY)

## OpenAI Batch Processing

In [ ]:
tariffs_transcripts = openai_batch_process(
    key = OPENAI_KEY,
    query = 'US Tariffs',
    filename = 'transcripts.csv'
    )
tariffs_transcripts.process_batch(
    id_field='videoId',
    text_field='transcript'
)

In [ ]:
tariffs_transcripts.retrieve_batch()

In [ ]:
tariffs_transcripts.download_output()

In [ ]:
with open('data/us-tariffs_transcripts.jsonl', 'r', encoding='utf-8') as file:
    filter_ids = [i['custom_id'] for i in [json.loads(line) for line in file if line.strip()]]

In [ ]:
tariffs_commentThreads = openai_batch_process(
    key = OPENAI_KEY,
    query = 'US Tariffs',
    filename = 'commentThreads.csv'
    )
tariffs_commentThreads.process_batch(
    id_field='id',
    text_field='textOriginal',
    filter_ids=filter_ids,
    filter_col='videoId'
)

In [ ]:
tariffs_commentThreads.retrieve_batch()

In [ ]:
tariffs_commentThreads.download_output()

In [ ]:
tariffs_commentThreadsreplies = openai_batch_process(
    key = OPENAI_KEY,
    query = 'US Tariffs',
    filename = 'commentThreadsreplies.csv'
    )
tariffs_commentThreadsreplies.process_batch(
    id_field='id',
    text_field='textOriginal',
    filter_ids=filter_ids,
    filter_col='videoId'
)

In [ ]:
tariffs_commentThreadsreplies.retrieve_batch()

In [ ]:
tariffs_commentThreadsreplies.download_output()

## Consolidate Datasets

In [ ]:
def reshape_download(
    query:str,
    filename:str,
    bucket_name:str = 'youtube-us-tariffs2'
) -> dict:
    # Get list of files
    blobnames = list_from_gcs(bucket_name, '-'.join(query.split()).lower(), filename)

    # Relevant cols
    cols = {
        'search.csv': [
            'videoId', 'channelId', 'channelTitle', 'title', 'description',
            'publishTime', 'publishedAt', 'liveBroadcastContent'
            ],
        'videos.csv': [
                'id', 'viewCount', 'favoriteCount', 'likeCount', 'commentCount', 'topicCategories'
            ],
        'commentThreads.csv': [
                'id', 'channelId', 'videoId', 'authorDisplayName', 
                'textDisplay', 'totalReplyCount', 'likeCount', 
                'updatedAt', 'publishedAt'
            ],
        'commentThreadsreplies.csv': [
                'id', 'parentId', 'channelId', 'videoId', 'authorDisplayName',
                'textDisplay', 'likeCount', 
                'updatedAt', 'publishedAt'
            ],
        'transcripts.csv': [
                'videoId', 'language', 'is_generated', 'transcript'
            ]
    }
    id_col = 'id' if filename == 'videos.csv' else 'videoId'
    
    # Download from GCS
    ids = set()
    all_data = []
    for blob in tqdm(blobnames, desc="Downloading blobs", unit="blob", total=len(blobnames)):
        data = load_from_gcs(bucket_name, blob_name=blob)
        if data:
            all_data += [x for x in data if x[id_col] in filter_ids]

    # Clean data
    seen = set()
    cleaned_data = []

    for item in all_data:
        key = item[id_col]
        if key not in seen:
            seen.add(key)
            cleaned_data.append({k: v for k, v in item.items() if k in cols[filename]})

    # Save as JSON
    if cleaned_data:
        with open(f"data/{filename.replace('.csv', '')}.json", 'w') as f:
            json.dump(cleaned_data, f, indent=2)

In [ ]:
query = 'US Tariffs'

reshape_download(query, 'search.csv')
reshape_download(query, 'videos.csv')
reshape_download(query, 'commentThreads.csv')
reshape_download(query, 'commentThreadsreplies.csv')
reshape_download(query, 'transcripts.csv')

In [ ]:
import aiohttp
import asyncio
import copy

async def fetch_with_retries(url, params, retry_limit=3, retry_delay=1, session=aiohttp.ClientSession(), verbose=False):
    """
    Fetch data from a URL with retries and handle errors related to disabled comments.
    
    Args:
        url (str): The URL to fetch data from.
        params (dict): Parameters to include in the request.
        retry_limit (int): The number of retries to attempt.
        retry_delay (int): The delay between retries in seconds.
        session (aiohttp.ClientSession): The session used to make HTTP requests.
        verbose (bool): Print verbose output. Default=False
        
    Returns:
        tuple: A tuple containing the response data and the nextPageToken if available.
    
    Raises:
        Exception: If retries are exhausted and the request still fails.
    """
    __params__ = copy.deepcopy(params)
    attempt = 0

    while attempt < retry_limit:
        try:
            async with session.get(url, params=__params__) as response:
                # Handle quota exceeded case
                if response.status == 403 and response.reason == 'Quota exceeded':
                    print(f"API quota exceeded")
                    return None, None

                response_data = await response.json()
                
                # Handle comments disabled case
                if response.status == 403 and 'disabled comments' in response_data['error'].get('message'):
                    if verbose:
                        print(f"Comments are disabled for video ID: {__params__['videoId']}")
                    return None, None
                
                # Handle API limit errors and other issues
                if response.status == 200:
                    next_page_token = response_data.get('nextPageToken')
                    return response_data, next_page_token
                else:
                    if verbose:
                        print(f"Received error response: {response_data}")
                    response.raise_for_status()
        
        except (aiohttp.ClientError, asyncio.TimeoutError) as e:
            attempt += 1
            if verbose:
                print(f"Attempt {attempt} failed: {e}. Retrying in {retry_delay} seconds...")
            await asyncio.sleep(retry_delay)
    
    # If all retries fail, raise an exception
    raise Exception(f"Failed to fetch data from {url} after {attempt} attempts.")

In [ ]:
import isodate
import json
with open('data/us-tariffs_transcripts.jsonl', 'r', encoding='utf-8') as file:
    ids = [i['custom_id'] for i in [json.loads(line) for line in file if line.strip()]]

url = 'https://www.googleapis.com/youtube/v3/videos'
params = {
    'part': 'id,contentDetails',
    'key': OPENAI_KEY
}
all_data = {}
closed_account = []
with tqdm(total=len(filter_ids), desc=f"Retrieving content details") as pbar:
    for i in filter_ids:
        params['id'] = i
        try:
            data = await fetch_with_retries(url, params)
            cleaned_data = data[0]['items'][0]
            all_data[cleaned_data['id']] = isodate.parse_duration(cleaned_data['contentDetails']['duration']).total_seconds()
        except:
            all_data[cleaned_data['id']] = -1

        pbar.update(1)